### Set up

In [3]:
import json
import os
import textwrap
import pandas as pd
import importlib
import numpy as np
import src.annotate_scenario as annotate_scenario
import src.prompts as prompts
import src.translate_to_vis as translate_to_vis
import src.node as node
import src.get_emb_distances as get_emb_distances
import src.utils as utils
from pathlib import Path
importlib.reload(annotate_scenario)
importlib.reload(translate_to_vis)

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Automatically reload modules when they change
%load_ext autoreload
%autoreload 2

In [ ]:
# set main paths
'''
CUR_DIR = os.path.dirname(os.path.abspath(__name__))
SCENARIO_DIR = CUR_DIR+'/scenarios/'
DATA_DIR_HUMAN = CUR_DIR+'/data/human_annotation/'
OUTPUT_DIR = CUR_DIR+'/annotated_outputs/'
'''
CUR_DIR = Path().resolve()
print(f"current_path: {CUR_DIR}")

SCENARIO_DIR = CUR_DIR / "scenarios"
DATA_DIR_HUMAN = CUR_DIR / "data" / "human_annotation"
OUTPUT_DIR = CUR_DIR / "annotated_outputs"

'\nCUR_DIR = Path().resolve()\nprint(f"current_path: {CUR_DIR}")\n\nSCENARIO_DIR = CUR_DIR / "scenarios"\nDATA_DIR_HUMAN = CUR_DIR / "data" / "human_annotation"\nOUTPUT_DIR = CUR_DIR / "annotated_outputs"\n'

## Look at Scenario

In [4]:
#set scenario file filename
FILENAME = 'scenarios.json'

#select scenario and action choice
SCENARIO_ID = 2
ACT_ID = '2'

#read in the scenario
scenario_json = utils.open_scenario(SCENARIO_DIR, FILENAME, SCENARIO_ID, ACT_ID)


Scenario Text: 


About three weeks ago, my girlfriend was driving while I was a passenger. We were at a four way stop
and if I'm being totally honest, I think she went when it wasn't her turn but it was really close
and her car and the other guys hit each other at an angle so it was really hard to tell who was at
fault. My girlfriend is about 3/4 African American but still somewhat light skinned. When we got out
of the car, the other guy was an older man who was clearly agitated but as god as my witness I did
not hear him say anything rude, let alone derogatory towards my girlfriend's race. He had called the
police even before getting out of the car and I think it took them less than 10 minutes to get
there. My girlfriend sat in her car calling her parents while the other guy and I did our best to
redirect traffic around the two cars. When the police came up my girlfriend instantly went from calm
and collected to tears streaming down her face when they asked her what happened she said

#### There are 4 major stages of processing.

0. Entities
Label the entities (no human data)

1. Value Scores / "Deontology"
Score the action in moral value

2. Outcomes
Map action to probable outcomes 

3. Outcome Utilities
Consequentialist analysis of harms/benefits of each outcome to each entity

4. Outcome Links
Connection between each entity and each outcome, in terms of Cause, Intend, and Desire



## Go through annotation process step by step (replicates main function in annotate_scenario) 

#### Set up

In [5]:

# get the action choice and convert to two pronoun options (I and Ziv)
this_act = scenario_json['options'][ACT_ID]
this_act_I = "I decide to " + this_act
this_act_Ziv = annotate_scenario.prompts.convert_I_Ziv(this_act_I)
print('\n\nAction choice:') 
print(this_act_Ziv)
print(this_act_I)

#get the scenario and convert to two pronoun options
this_scenario = scenario_json['text']
this_scenario_Ziv = annotate_scenario.prompts.convert_I_Ziv(this_scenario)


# create a dictionary to write out to csv later
scenario_dict = {'scenario': this_scenario, 'scenario_idx': scenario_json['id'],
                    'choice': this_act_I}





Action choice:
Ziv decides to say nothing and not contradict their girlfriend.
I decide to say nothing and not contradict my girlfriend


In [6]:

#initialize Graph object    
g = annotate_scenario.node.Graph()
g.reset()   
print('Graph g initialized and reset.')

Graph g initialized and reset.


#### Step 0. Get entities

In [7]:
#Step 0. Get entities

# identify all sentient beings, returning both pronoun forms and a string list
returned_beings = annotate_scenario.process_beings(this_scenario,this_act,g)
beings_I = returned_beings[0]
beings_Ziv = returned_beings[1]
beings_str_list = returned_beings[2]

#update the scenario dict with the beings
scenario_dict["entities"] = beings_str_list





Identified these entities: 

I
my girlfriend
the other guy
the police officers
my girlfriend's parents


#### Step 1. Deontology / Action Value Scores

In [8]:
#Step 1.  #ACTION VALUE SCORES

#call the process_values function to rate the moral goodness or wrongness of the action with no context
processed_values  = annotate_scenario.process_value_simple(this_act,this_act_I,g) 

print(processed_values)

    

You are an expert judge of the moral virtues of different actions. Moral actions are those which humans consider virtuous, that consider others' wellbeing and happiness, and that are guided by principles of ethics. You will be given a sentence describing an action. You must rate thus action in terms of its moral virtue. Respond with a rating from -100 to 100 for this action, where -100 is the most morally wrong action you can possibly think of, and 100 is the most morally virtuous action you can possibly think of. 0 indicates a morally neutral action. Please return a json object with key:value pair of "score": rating. Please rate this action: I decide to say nothing and not contradict my girlfriend
{'score': 0}


#### Step 2. Outcome Likelihoods

In [9]:
#Step 2. Outcomes

processed_events = annotate_scenario.process_outcomes(this_scenario, this_act)
events_I= processed_events[1]
events_Ziv= processed_events[0]
print("\n".join(events_I))         
scenario_dict["outcomes"]= events_I

My girlfriend's account is accepted as uncontested by me.
The police record my girlfriend's statement as the primary witness account.
The older man is accused of making racist remarks
The older man experiences distress from the accusation
I experience internal conflict or guilt for not speaking up.
My girlfriend feels supported by my silence.
The police may treat the older man as a potential perpetrator of a hate incident
The investigation proceeds based on the information provided by my girlfriend.


#### Step 3. Outcome Utilities

In [10]:
#Step 3. Outcome utilities

impacts_list = annotate_scenario.process_impacts(this_scenario_Ziv, this_act, this_act_Ziv, events_Ziv, events_I,beings_Ziv,g) 



Processing impacts of event: My girlfriend's account is accepted as uncontested by me.
Scored impacts for these beings:
['Ziv', "Ziv's girlfriend", 'the other guy', 'the police officers', "Ziv's girlfriend's parents"]
Scored values:
[-30, 30, -80, 10, 20]

Processing impacts of event: The police record my girlfriend's statement as the primary witness account.
Scored impacts for these beings:
['Ziv', "Ziv's girlfriend", 'the other guy', 'the police officers', "Ziv's girlfriend's parents"]
Scored values:
[-20, 20, -60, -10, 10]

Processing impacts of event: The older man is accused of making racist remarks
Scored impacts for these beings:
['Ziv', "Ziv's girlfriend", 'the other guy', 'the police officers', "Ziv's girlfriend's parents"]
Scored values:
[-30, 20, -80, -10, 10]

Processing impacts of event: The older man experiences distress from the accusation
Scored impacts for these beings:
['Ziv', "Ziv's girlfriend", 'the other guy', 'the police officers', "Ziv's girlfriend's parents"]
S

#### Step 4. Cause / Intend / Know Links

In [11]:
#Step 4. causal / intentional / knowledge links -- run on currently generated event/outcome list
output_links = annotate_scenario.process_causal_links(this_scenario_Ziv, events_Ziv, events_I, this_act_Ziv,g)    


Processing event: My girlfriend's account is accepted as uncontested by me.
{'cause': 'yes', 'intend': 'yes', 'know': 'yes'}
CKI links for I
C+I+K+

Processing event: The police record my girlfriend's statement as the primary witness account.
{'cause': 'yes', 'intend': 'no', 'know': 'yes'}
CKI links for I
C+I-K+

Processing event: The older man is accused of making racist remarks
{'cause': 'yes', 'intend': 'no', 'know': 'yes'}
CKI links for I
C+I-K+

Processing event: The older man experiences distress from the accusation
{'cause': 'yes', 'intend': 'no', 'know': 'yes'}
CKI links for I
C+I-K+

Processing event: I experience internal conflict or guilt for not speaking up.
{'cause': 'yes', 'intend': 'no', 'know': 'yes'}
CKI links for I
C+I-K+

Processing event: My girlfriend feels supported by my silence.
{'cause': 'yes', 'intend': 'no', 'know': 'no'}
CKI links for I
C+I-K-

Processing event: The police may treat the older man as a potential perpetrator of a hate incident
{'cause': 'yes'

#### Step 5. Write out the results

In [12]:
#optional -- write out the results 

this_output_filename = f"{OUTPUT_DIR}scenarios_{SCENARIO_ID}_choice_{ACT_ID}.json"
print('\n\nWriting to file: '+this_output_filename)
g_print = g.print_graph()
utils.write_jsonlines(this_output_filename,g_print)
print('\n\n')


translate_to_vis.main(this_output_filename)




Writing to file: /Users/rylenc/Dropbox/2025_moral_scenario_annotation/code/rylen/graph_extract/annotated_outputs/scenarios_2_choice_2.json



/Users/rylenc/Dropbox/2025_moral_scenario_annotation/code/rylen/graph_extract/annotated_outputs/scenarios_2_choice_2.json
